# Connection Check & Catalog Discovery

This notebook:
1. Confirms AWS credentials work.
2. Lists the Glue databases and tables (so we see the *current* names).
3. Pulls a tiny sample from the dimension tables.
4. Locates the live telemetry (fact) table and peeks at the compliance results.

**Before running:** make sure you have logged in and selected the profile:

```powershell
aws sso login --profile ciccada
$env:AWS_PROFILE = "ciccada"
```

## 1. Credentials check

In [4]:
import boto3
session = boto3.Session(profile_name="ciccada", region_name="ap-southeast-2")
ident = session.client("sts").get_caller_identity()
print("Account:", ident["Account"])
print("Identity:", ident["Arn"])
# If this errors with 'Unable to locate credentials', run `aws sso login` and
# set AWS_PROFILE, then restart the kernel.

Account: 130340360668
Identity: arn:aws:sts::130340360668:assumed-role/AWSReservedSSO_AWSAdministratorAccess_58ece215f84a4b54/z3553082_sa@ad.unsw.edu.au


## 2. The catalog
Mapping (`SolA_ts4`, `SolA_circuits`) to whatever they are called today.

In [ ]:
from aws_config import databases, tables, aq, columns
databases()

,Database,Description
0,bom_nci,
1,elb_logdb,
2,sapn2022,
3,solar_analytics,Migrated from Hive Metastore
4,solar_analytics_iceberg,
5,test_db,
6,type_probe,


In [ ]:
# Tables in the main analytics database
tables("solar_analytics")[["Database", "Table", "TableType"]]

,Database,Table,TableType
0,solar_analytics,circuits,EXTERNAL_TABLE
1,solar_analytics,compliance_voltvar,EXTERNAL_TABLE
2,solar_analytics,compliance_voltwatt,EXTERNAL_TABLE
3,solar_analytics,meta_single_inverters,EXTERNAL_TABLE
4,solar_analytics,meta_single_inverters_wrong_capacity,EXTERNAL_TABLE
5,solar_analytics,meta_single_inverters_wrong_capacity_up2_3c,EXTERNAL_TABLE
6,solar_analytics,partition_lookup,EXTERNAL_TABLE
7,solar_analytics,raw_bom_2024_1,EXTERNAL_TABLE
8,solar_analytics,sites,EXTERNAL_TABLE
9,solar_analytics,test_sola_2025_12,EXTERNAL_TABLE


In [ ]:
# The Iceberg database
tables("solar_analytics_iceberg")[["Database", "Table", "TableType"]]

,Database,Table,TableType
0,solar_analytics_iceberg,all_uncurtailedpv,EXTERNAL_TABLE
1,solar_analytics_iceberg,circuits,EXTERNAL_TABLE
2,solar_analytics_iceberg,conformance_antiisland,EXTERNAL_TABLE
3,solar_analytics_iceberg,conformance_sust_op,EXTERNAL_TABLE
4,solar_analytics_iceberg,conformance_sust_op_3w,EXTERNAL_TABLE
5,solar_analytics_iceberg,conformance_voltvar,EXTERNAL_TABLE
6,solar_analytics_iceberg,conformance_voltwatt,EXTERNAL_TABLE
7,solar_analytics_iceberg,conformance_voltwattghi,EXTERNAL_TABLE
8,solar_analytics_iceberg,meta_up23c,EXTERNAL_TABLE
9,solar_analytics_iceberg,pv_ghi_norm_model,EXTERNAL_TABLE


## 3. Sample the dimension tables

In [ ]:
aq("SELECT * FROM circuits LIMIT 5")

,site_id,device_id,circuit_id,device_type,circuit_polarity,circuit_type,is_pv
0,484720983,135961,84703,Watt Watcher,1,pv_site_net,True
1,484720983,135961,84704,Watt Watcher,1,pv_site_net,True
2,484720983,135961,84705,Watt Watcher,1,pv_site_net,True
3,150652475,140483,658777,Watt Watcher,1,ac_load_net,False
4,150652475,140483,658778,Watt Watcher,1,ac_load_net,False


In [ ]:
aq("SELECT * FROM sites LIMIT 5")

,site_id,state,postcode,longitude,latitude,dnsp_name,dc_capacity_kw,ac_capacity_kw,export_limit_kw,monitoring_start,inverter_count,pv_install_date,manufacturer,model,ac_capacity_kw_exploaded,installed_after_18_dec_2021
0,1944472430,NSW,2502.0,150.85,-34.470,Endeavour,5.18,5.0,3.0,2021-08-09,1.0,2021-08-09,Sungrow,SG5KTL,5.0,False
1,1555410224,QLD,4814.0,146.75,-19.305,Ergon,13.28,10.0,5.0,2024-03-07,1.0,2024-03-06,Sungrow,SG10RS-ADA,10.0,True
2,623277618,QLD,4211.0,153.30,-27.990,Energex,15.75,10.0,5.0,2024-03-18,1.0,2023-06-30,Fronius,Primo GEN24 10.0,10.0,True
3,1245528685,QLD,4078.0,152.95,-27.630,Energex,15.00,10.0,6.8,2024-09-18,1.0,2024-09-18,Generic Inverter,10.0kW,10.0,True
4,1651877625,NSW,2350.0,151.70,-30.510,Essential,11.55,9.6,5.0,2019-06-27,2.0,2018-11-02,Generic Inverter,5kW,5.0,False


In [ ]:
# The partition lookup tells you which (year, month) partitions actually exist
aq("SELECT * FROM partition_lookup LIMIT 20")

,year,month,is_pv
0,2024,1,False
1,2024,1,True
2,2024,10,False
3,2024,10,True
4,2024,11,False
5,2024,11,True
6,2024,12,False
7,2024,12,True
8,2024,2,False
9,2024,2,True


## 4. Find the live telemetry table

The colleague queried `SolA_ts4` / `SolA_ts5`. Look at the table lists printed
in section 2 and find the equivalent today (likely in `solar_analytics_iceberg`,
possibly named `sola_ts4`, `sola_ts5`, or `bucketed_table4`).

**Edit `FACT_TABLE` and `FACT_DB` below to the real name**, then run the cell.
Note the partition filters — never query this table without them.

In [ ]:
FACT_TABLE = "ts"                       # <-- 
FACT_DB    = "solar_analytics_iceberg"  # <-- 

# Just the schema first:
columns(FACT_TABLE, database=FACT_DB)

,Column Name,Type,Partition,Comment
0,circuit_id,bigint,False,
1,t_stamp,timestamp,False,
2,power,double,False,
3,energy,double,False,
4,energy_reactive,double,False,
5,energy_import,double,False,
6,energy_export,double,False,
7,energy_reactive_import,double,False,
8,energy_reactive_export,double,False,
9,power_factor,double,False,


In [ ]:
# partition-pruned sample
YEAR = 2025
MONTH = 1
sample = aq(f'''
    SELECT circuit_id, t_stamp, voltage, power, energy_reactive
    FROM {FACT_TABLE}
    WHERE is_pv = True AND year = {YEAR} AND month = {MONTH}
      AND circuit_id = 547781
    ORDER BY t_stamp
    LIMIT 20
''', database=FACT_DB)
sample

,circuit_id,t_stamp,voltage,power,energy_reactive
0,547781,2025-01-01 00:00:00,240.75,12932.1900,-1564.3222
1,547781,2025-01-01 00:05:00,241.05,13119.8867,-1585.2928
2,547781,2025-01-01 00:10:00,241.10,13208.9300,-1605.0239
3,547781,2025-01-01 00:15:00,240.30,13364.4233,-1620.2044
4,547781,2025-01-01 00:20:00,240.75,13596.0833,-1649.4522
5,547781,2025-01-01 00:25:00,240.85,13693.9867,-1677.1553
6,547781,2025-01-01 00:30:00,240.90,13849.0100,-1688.7728
7,547781,2025-01-01 00:35:00,240.55,13987.6467,-1715.7594
8,547781,2025-01-01 00:40:00,240.15,13996.4733,-1716.3953
9,547781,2025-01-01 00:45:00,240.60,14090.7067,-1726.9397


## 5. Analysis outputs

These tables are the *results* the report was built from. Their schemas tell us
exactly what the `SolA2024_Analysis` notebooks ultimately produced — very useful
for understanding the pipeline.

In [ ]:
aq("SELECT * FROM compliance_voltwatt LIMIT 5")

,site_id,s_id,year,month,day,noncompliance_voltwatt_count,noncompliance_voltwatt_sum,total_count
0,465008538,S3443,2025,3,5,81,266.511457,127
1,465008538,S3443,2025,3,6,76,315.396067,143
2,1837079785,S13543,2025,3,2,74,334.664850,87
3,1837079785,S13543,2025,3,3,70,364.471854,82
4,465008538,S3443,2025,3,13,70,298.412619,123


In [ ]:
aq("SELECT circuit_id, site_id, state FROM compliance_voltvar LIMIT 5")

QueryFailed: HIVE_BAD_DATA: Malformed Parquet file. Field q_adverse_count's type INT64 in parquet file s3a://project-ciccada/spark-warehouse/Compliance_results_SolA/compliance_voltvar.parquet/part-00003-39357387-becd-4f82-b247-0f8075084ed6-c000.snappy.parquet is incompatible with type double defined in table schema [s3a://project-ciccada/spark-warehouse/Compliance_results_SolA/compliance_voltvar.parquet/part-00003-39357387-becd-4f82-b247-0f8075084ed6-c000.snappy.parquet]

In [ ]:
# What inverter metadata was inferred (nameplate capacity, etc.)
aq("SELECT * FROM meta_single_inverters LIMIT 5")

,circuit_id,site_id,device_id,device_type,circuit_polarity,circuit_type,is_pv,state,postcode,longitude,...,dc_capacity_kw,ac_capacity_kw,export_limit_kw,monitoring_start,inverter_count,pv_install_date,manufacturer,model,ac_capacity_kw_exploaded,installed_after_18_dec_2021
0,5439,54785239,88964,Watt Watcher,1,pv_site,True,NSW,2038.0,151.15,...,1.02,1.0,NaN,2015-07-20,1.0,2007-01-01,SMA,Sunny Boy SB 1100,1.0,False
1,14275,1677299261,89052,Watt Watcher,1,pv_site_net,True,QLD,4215.0,153.40,...,3.08,3.0,NaN,2016-02-03,1.0,2015-12-16,ABB,PVI-3.0-TL-OUTD,3.0,False
2,14422,89943339,89078,Watt Watcher,1,pv_site_net,True,NSW,2260.0,151.40,...,4.16,4.0,NaN,2016-02-05,1.0,2016-01-15,Solis,Solis-4K-2G,4.0,False
3,9991,1876455596,89040,Watt Watcher,1,pv_site_net,True,VIC,3084.0,145.05,...,3.15,3.0,NaN,2015-12-08,1.0,2015-11-10,ABB,PVI-3.0-TL-OUTD,3.0,False
4,11275,1134579538,180500,Watt Watcher,1,pv_site_net,True,QLD,4207.0,153.25,...,15.00,15.0,NaN,2017-06-26,1.0,2016-01-21,SMA,Sunny Tripower15000 TL,15.0,False
